# Text Analytics Coursework

This notebook provides some example code for loading and examining the dataset for task 2. 

In [1]:
%load_ext autoreload
%autoreload 2

# Use HuggingFace's datasets library to access the Emotion dataset
from datasets import load_dataset
import numpy as np
import pandas as pd

# Task 2 - EBM-NLP

This dataset is provided at https://github.com/bepnye/EBM-NLP and a copy has been made available in this repository for convenience. The data will need to be unzipped:

In [2]:
import tarfile
import os

path_tofile = "./ebm_nlp_2_00.tar.gz"
extract_directory = os.path.dirname(path_tofile)

if tarfile.is_tarfile(path_tofile):
    with tarfile.open(path_tofile) as f:
        f.extractall(path=extract_directory)  # Extract all members from the archive to the current working directory


The data contains text documents that are annotated for mentions of participants, interventions and outcomes (PIO) in medical research. For each entity type, P, I, or O, there is a slightly different set of documents in the training and test set. Most of the documents are identical, but each type has a few extra documents. So, let's deal with each type separately for now.

To load the text documents, we first make a list of the document IDs for one entity type (P, I or O):

In [3]:
from pathlib import Path

DATA_DIR = Path("./ebm_nlp_2_00")

docs_dir = DATA_DIR / "documents"

def get_doc_ids(split="train", label_type="participants"):
    """ 
    split: 'train' or 'test' 
    """

    if split == "test":
        split = "test/gold"

    train_dir = (
        DATA_DIR
        / "annotations"
        / "aggregated"
        / "hierarchical_labels"
        / label_type  # assuming that the split is the same for all entity types, we can just look at one of them
        / split
    )
    
    doc_ids = [p.stem.split(".")[0] for p in train_dir.glob("*.AGGREGATED.ann")]
   # print(doc_ids)
    return sorted(doc_ids)

doc_ids_p = get_doc_ids("train", "participants")
test_doc_ids_p = get_doc_ids("test", "participants")

print(f"Number of documents in train split for participants: {len(doc_ids_p)}")
print(f"Number of documents in test split for participants: {len(test_doc_ids_p)}")


Number of documents in train split for participants: 4609
Number of documents in test split for participants: 189


Now, we can get the annotations for the first entity type:

In [4]:
def load_labels_for_doc(doc_id, label_type="participants", split="train"):
    """
    label_type: 'participants', 'interventions', or 'outcomes'
    split: 'train' or 'test' 
    """
    if split == "test":
        split = "test/gold"

    ann_path = DATA_DIR / "annotations" / "aggregated" / "hierarchical_labels" / label_type / split/ f"{doc_id}.AGGREGATED.ann"
    
    if not ann_path.exists():
        print(ann_path, "does not exist!")
        return None
    
    with open(ann_path, "r", encoding="utf-8") as f:
        labels = [line.strip() for line in f]
    
    return labels

def load_labels(doc_ids, label_type="participants", split="train"):
    labels = []
    for doc_id in doc_ids:
        doc_labels = load_labels_for_doc(doc_id, label_type, split)
        if doc_labels is not None:
            labels.append(doc_labels)
    return labels

participants_labels = load_labels(doc_ids_p, "participants", split="train")

print(f"Length of participants_labels: {len(participants_labels)}")

test_participants_labels = load_labels(test_doc_ids_p, "participants", split="test")
print(f"Length of test_participants_labels: {len(test_participants_labels)}")

sample = 123
print("Document ID:", doc_ids_p[sample])
print(f"Participants label example for doc {doc_ids_p[sample]}:")
print(participants_labels[sample])

Length of participants_labels: 4609
Length of test_participants_labels: 189
Document ID: 10674680
Participants label example for doc 10674680:
['0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '4', '4', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '1', '1', '1', '1', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '

In [5]:
from itertools import chain
import numpy as np

all_labels = chain(*participants_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels)))

['0' '1' '2' '3' '4']


Let's look at what labels there are for Participants. The code above shows there are four values: 0 corresponds to 'outside' but 1-4 all indicate tokens that form an entity span. Each number is a level in a hierarchy of specificity. To start with let's not worry about this 'hierarchy'. We can instead just turn the labels into simple BIO (Beginning of a span, Inside a span, and Outside a span) tags.

In [6]:
def hierarchical_to_bio(tags):
    """
    Convert EBM-NLP hierarchical labels (0–4) to flat BIO tags.

    Parameters
    ----------
    tags : list[int]
        A list of hierarchical labels for a single document.

    Returns
    -------
    list[str]
        BIO tags ("O", "B", "I").
    """

    bio = []
    prev = 0

    for t in tags:
        t = int(t)  # ensure it's an integer
        if t == 0:
            bio.append("O")
        else:
            if prev == 0:
                bio.append("B")
            else:
                bio.append("I")

        prev = t
        

    return bio

def convert_all_labels_to_bio(labels):
    for i, doc_labels in enumerate(labels):
        labels[i] = hierarchical_to_bio(doc_labels)
    return labels

participants_labels = convert_all_labels_to_bio(participants_labels)
test_participants_labels = convert_all_labels_to_bio(test_participants_labels)

all_labels = chain(*participants_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels)))

['B' 'I' 'O']


So far, we've loaded the document IDs for participants and the corresponding labels. Now, let's load the documents themselves. They are already tokenised so that the labels match up with the tokens:

In [7]:
def load_document(doc_id):
    doc_path = DATA_DIR / "documents" / f"{doc_id}.tokens"
    with open(doc_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

def load_documents(doc_ids):
    documents = []
    for doc_id in doc_ids:
        doc = load_document(doc_id)
        documents.append(doc)
    return documents

participants_tokens = load_documents(doc_ids_p)
test_participants_tokens = load_documents(test_doc_ids_p)

# inspect a random element
print("Document ID:", doc_ids_p[sample])
print(f"Tokenised document example for doc {doc_ids_p[sample]}:")
print(participants_tokens[sample])
print(participants_labels[sample])

Document ID: 10674680
Tokenised document example for doc 10674680:
['Assessment', 'of', 'therapeutic', 'response', 'of', 'Plasmodium', 'falciparum', 'to', 'chloroquine', 'and', 'sulfadoxine-pyrimethamine', 'in', 'an', 'area', 'of', 'low', 'malaria', 'transmission', 'in', 'Colombia', '.', 'Although', 'chloroquine', '(', 'CQ', ')', 'resistance', 'was', 'first', 'reported', 'in', 'Colombia', 'in', '1961', 'and', 'sulfadoxine-pyrimethamine', '(', 'SP', ')', 'resistance', 'in', '1981', ',', 'the', 'frequency', 'of', 'treatment', 'failures', 'to', 'these', 'drugs', 'in', 'Colombia', 'is', 'unclear', '.', 'A', 'modified', 'World', 'Health', 'Organization', '14-day', 'in', 'vivo', 'drug', 'efficacy', 'test', 'for', 'uncomplicated', 'Plasmodium', 'falciparum', 'malaria', 'in', 'areas', 'with', 'intense', 'malaria', 'transmission', 'was', 'adapted', 'to', 'reflect', 'the', 'clinical', 'and', 'epidemiologic', 'features', 'of', 'a', 'low-intensity', 'malaria', 'transmission', 'area', 'in', 'the', 

### Interventions

In [8]:
doc_ids_i = get_doc_ids("train", "interventions")
test_doc_ids_i = get_doc_ids("test", "interventions")

print(f"Number of documents in train split for interventions: {len(doc_ids_i)}")
print(f"Number of documents in test split for interventions: {len(test_doc_ids_i)}")

interventions_tokens = load_documents(doc_ids_i)
test_interventions_tokens = load_documents(test_doc_ids_i)

Number of documents in train split for interventions: 4746
Number of documents in test split for interventions: 187


In [9]:
interventions_labels = load_labels(doc_ids_i, "interventions", split="train")
print(f"Length of interventions_labels: {len(interventions_labels)}")

test_interventions_labels = load_labels(test_doc_ids_i, "interventions", split="test")
print(f"Length of test_interventions_labels: {len(test_interventions_labels)}")

interventions_labels = convert_all_labels_to_bio(interventions_labels)
test_interventions_labels = convert_all_labels_to_bio(test_interventions_labels)

Length of interventions_labels: 4746
Length of test_interventions_labels: 187


### Outcomes

In [10]:
doc_ids_o = get_doc_ids("train", "outcomes")
test_doc_ids_o = get_doc_ids("test", "outcomes")

print(f"Number of documents in train split for outcomes: {len(doc_ids_o)}")
print(f"Number of documents in test split for outcomes: {len(test_doc_ids_o)}")

outcomes_tokens = load_documents(doc_ids_o)
test_outcomes_tokens = load_documents(test_doc_ids_o)

Number of documents in train split for outcomes: 4681
Number of documents in test split for outcomes: 190


In [11]:
outcomes_labels = load_labels(doc_ids_o, "outcomes", split="train")
print(f"Length of outcomes_labels: {len(outcomes_labels)}")

test_outcomes_labels = load_labels(test_doc_ids_o, "outcomes", split="test")   
print(f"Length of test_outcomes_labels: {len(test_outcomes_labels)}")

outcomes_labels = convert_all_labels_to_bio(outcomes_labels)
test_outcomes_labels = convert_all_labels_to_bio(test_outcomes_labels)


Length of outcomes_labels: 4681
Length of test_outcomes_labels: 190


In [12]:
print(f"Intersection of train doc IDs across entity types: {len(set(doc_ids_p) & set(doc_ids_i) & set(doc_ids_o))}")
print(f"Intersection of test doc IDs across entity types: {len(set(test_doc_ids_p) & set(test_doc_ids_i) & set(test_doc_ids_o))}")
print(f"Documents that are different across entity types in train split: {len((set(doc_ids_p) | set(doc_ids_i) | set(doc_ids_o)) - (set(doc_ids_p) & set(doc_ids_i) & set(doc_ids_o)))}")
print(f"Documents that are different across entity types in test split: {len((set(test_doc_ids_p) | set(test_doc_ids_i) | set(test_doc_ids_o)) - (set(test_doc_ids_p) & set(test_doc_ids_i) & set(test_doc_ids_o)))}")


print(f"Test examples of the participants type that are in other entity types' training splits: {set(test_doc_ids_p) & (set(doc_ids_i) | set(doc_ids_o))}")
print(f"Test examples of the interventions type that are in other entity types' training splits: {set(test_doc_ids_i) & (set(doc_ids_p) | set(doc_ids_o))}")
print(f"Test examples of the outcomes type that are in other entity types' training splits: {set(test_doc_ids_o) & (set(doc_ids_p) | set(doc_ids_i))}")

Intersection of train doc IDs across entity types: 4457
Intersection of test doc IDs across entity types: 184
Documents that are different across entity types in train split: 344
Documents that are different across entity types in test split: 7
Test examples of the participants type that are in other entity types' training splits: set()
Test examples of the interventions type that are in other entity types' training splits: set()
Test examples of the outcomes type that are in other entity types' training splits: set()


### Interventions Embeddings

In [13]:
from gensim.models import word2vec
from gensim.utils import tokenize

In [14]:
interventions_tokens[0]

['[',
 'Triple',
 'therapy',
 'regimens',
 'involving',
 'H2',
 'blockaders',
 'for',
 'therapy',
 'of',
 'Helicobacter',
 'pylori',
 'infections',
 ']',
 '.',
 'Comparison',
 'of',
 'ranitidine',
 'and',
 'lansoprazole',
 'in',
 'short-term',
 'low-dose',
 'triple',
 'therapy',
 'for',
 'Helicobacter',
 'pylori',
 'infection',
 '.',
 'To',
 'evaluate',
 'the',
 'efficacy',
 'and',
 'safety',
 'of',
 'two',
 '1-week',
 'low-dose',
 'triple-therapy',
 'drug',
 'regimens',
 'involving',
 'antisecretory',
 'drugs',
 'for',
 'Helicobacter',
 'pylori',
 'infection',
 ',',
 '99',
 'patients',
 'with',
 'H.',
 'pylori',
 'infection',
 'were',
 'treated',
 'with',
 'either',
 'lansoprazole',
 '(',
 'LPZ',
 ')',
 'or',
 'ranitidine',
 '(',
 'RNT',
 ')',
 'used',
 'together',
 'with',
 'clarithromycin',
 '(',
 'CAM',
 ')',
 'and',
 'metrinidazole',
 '(',
 'MTZ',
 ')',
 '.',
 'The',
 'drug',
 'combination',
 'and',
 'administration',
 'periods',
 'in',
 'the',
 'PPI',
 'group',
 'were',
 'LPZ',
 

In [15]:
emb_interventions_model = word2vec.Word2Vec(interventions_tokens, sg=1, min_count=1, window=5, vector_size=100)

In [16]:
emb_interventions_model.wv.similar_by_word("ranitidine", topn=20)

[('omeprazole', 0.91133713722229),
 ('valacyclovir', 0.9023220539093018),
 ('alfa', 0.898934006690979),
 ('dalteparin', 0.8954160809516907),
 ('indomethacin', 0.8950571417808533),
 ('metoclopramide', 0.8926854729652405),
 ('lansoprazole', 0.8924328088760376),
 ('metronidazole', 0.8914836645126343),
 ('ketorolac', 0.8912112712860107),
 ('propionate', 0.888921856880188),
 ('megestrol', 0.8882537484169006),
 ('esomeprazole', 0.8866807818412781),
 ('gentamicin', 0.8848816752433777),
 ('famotidine', 0.8837569952011108),
 ('bismuth', 0.8794000148773193),
 ('metoprolol', 0.8792323470115662),
 ('amoxycillin', 0.8771522641181946),
 ('filgrastim', 0.8770647048950195),
 ('codeine', 0.8769855499267578),
 ('cephalexin', 0.8768534064292908)]

In [17]:
emb_interventions_model.wv.has_index_for("thingy")

False

In [18]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [19]:
len(max(interventions_tokens))

318

In [20]:
def pad_lists(lists, max_length, fill_value):
    padded_list = []
    for l in lists:
        if len(l) < max_length:
           l1 = l + [fill_value] * (max_length - len(l))
        padded_list.append(l1)
    return padded_list

In [21]:
padded_interventions_tokens = pad_lists(interventions_tokens, 350, "0")

In [22]:
import torch
from torch import tensor

In [23]:
#lengths = 0
#for i, l in enumerate(padded_interventions_tokens):
#        print(i, len(l))

In [24]:
padded_interventions_labels = pad_lists(interventions_labels, 350, "O")

In [25]:
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [26]:
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3)

for param in model.base_model.parameters():
    param.requires_grad = False

for param in model.bert.pooler.parameters():
    param.requires_grad = True

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [27]:
import evaluate

accuracy_metric = evaluate.load("accuracy")

In [28]:
from transformers import AutoTokenizer, DistilBertForTokenClassification
import torch

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
model = DistilBertForTokenClassification.from_pretrained("distilbert/distilbert-base-uncased")

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [29]:
token1 = interventions_tokens[0][17]
tokenizer.add_tokens(token1)

1

In [30]:
token1_label = interventions_labels[0][17]
token1_label

'B'

In [31]:
tokenizer.add_tokens("hello")

1

In [32]:
len(tokenizer.vocab)

30523

In [33]:
for item in list(tokenizer.vocab.items()):
    if item[0] == "ranitidine":
        print(item)

('ranitidine', 30522)


In [34]:
token_test2 = tokenizer.add_tokens(interventions_tokens[0])
token_test2

77

In [35]:
its = tokenizer.vocab.items()

In [37]:
interventions_initial_labels = load_labels(doc_ids_i, "interventions", split="train")
print(interventions_initial_labels[0])

['0', '0', '0', '0', '0', '3', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '3', '3', '3', '3', '0', '3', '3', '3', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '3', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '3', '0', '0', '0', '3', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0']


In [65]:
zipped_labels = []
for i in range(0, len(interventions_labels)):
    zipped_labels.append(list(zip(interventions_labels[i], interventions_initial_labels[i])))
zipped_labels[1]

[('B', '3'),
 ('O', '0'),
 ('O', '0'),
 ('O', '0'),
 ('O', '0'),
 ('O', '0'),
 ('O', '0'),
 ('O', '0')]

In [97]:
def combine_labels(zipped_label):
    combined_labels = []

    for item in zipped_label:
        if item[1] == '0':
            combined_labels.append("O")
        else:
            combined_labels.append((item[0] + "-" + item[1]))
            
    return combined_labels


def combine_all_labels(zipped_labels):
    all_combined_labels = []
    for item in zipped_labels:
        output = combine_labels(item)
        all_combined_labels.append(output)
    return all_combined_labels

In [99]:
intervention_combined_labels = combine_all_labels(zipped_labels)
all_labels = chain(*intervention_combined_labels)
print(np.unique(list(all_labels)))

['B-1' 'B-2' 'B-3' 'B-4' 'B-5' 'B-6' 'B-7' 'I-1' 'I-2' 'I-3' 'I-4' 'I-5'
 'I-6' 'I-7' 'O']


In [100]:
from transformers import AutoTokenizer, DistilBertForTokenClassification
import torch

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [102]:
print(interventions_tokens[0])

['[', 'Triple', 'therapy', 'regimens', 'involving', 'H2', 'blockaders', 'for', 'therapy', 'of', 'Helicobacter', 'pylori', 'infections', ']', '.', 'Comparison', 'of', 'ranitidine', 'and', 'lansoprazole', 'in', 'short-term', 'low-dose', 'triple', 'therapy', 'for', 'Helicobacter', 'pylori', 'infection', '.', 'To', 'evaluate', 'the', 'efficacy', 'and', 'safety', 'of', 'two', '1-week', 'low-dose', 'triple-therapy', 'drug', 'regimens', 'involving', 'antisecretory', 'drugs', 'for', 'Helicobacter', 'pylori', 'infection', ',', '99', 'patients', 'with', 'H.', 'pylori', 'infection', 'were', 'treated', 'with', 'either', 'lansoprazole', '(', 'LPZ', ')', 'or', 'ranitidine', '(', 'RNT', ')', 'used', 'together', 'with', 'clarithromycin', '(', 'CAM', ')', 'and', 'metrinidazole', '(', 'MTZ', ')', '.', 'The', 'drug', 'combination', 'and', 'administration', 'periods', 'in', 'the', 'PPI', 'group', 'were', 'LPZ', '30', 'mg', ',', 'CAM', '400', 'mg', ',', 'MTZ', '500', 'mg', '(', 'LCM', 'group', ')', '.', 'T

In [ ]:
tokenized_interventions = tokenizer(interventions_tokens, is_split_into_words=True)

### Misaligned Data
Here we can see that the tokenizer has split several of the words into multiple tokens. This will misalign the tokens with the labels so we need to fix this issue
The best way to do this is to use the tokenizers inbuilt word_ids method which tracks the indices of each partial token so that we can add the correct label to each of the tokens

In [110]:
print(tokenized_interventions.word_ids(batch_index=0))

[None, 0, 1, 2, 3, 3, 4, 5, 5, 6, 6, 7, 8, 9, 10, 10, 10, 10, 10, 11, 11, 11, 12, 13, 14, 15, 16, 17, 17, 17, 18, 19, 19, 19, 19, 20, 21, 21, 21, 22, 22, 22, 23, 24, 25, 26, 26, 26, 26, 26, 27, 27, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 38, 38, 39, 39, 39, 40, 40, 40, 41, 42, 42, 43, 44, 44, 44, 44, 45, 46, 47, 47, 47, 47, 47, 48, 48, 48, 49, 50, 51, 52, 53, 54, 54, 55, 55, 55, 56, 57, 58, 59, 60, 61, 61, 61, 61, 62, 63, 63, 64, 65, 66, 66, 66, 67, 68, 68, 69, 70, 71, 72, 73, 73, 73, 73, 73, 73, 74, 75, 76, 77, 78, 78, 78, 78, 78, 79, 80, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 91, 92, 93, 94, 94, 95, 96, 97, 98, 99, 100, 101, 102, 102, 103, 104, 105, 106, 106, 107, 108, 109, 110, 111, 111, 111, 112, 113, 114, 114, 115, 116, 117, 118, 119, 120, 121, 122, 122, 123, 124, 125, 126, 126, 127, 128, 129, 130, 131, 132, 133, 134, 134, 135, 135, 135, 136, 137, 138, 139, 140, 141, 142, 142, 143, 144, 145, 146, 147, 148, 148, 148, 149, 150, 151, 152, 153, 154, 154, 155, 156, 157

In [114]:
from datasets import Dataset

As we can see above several of the indices are repeated showing the words have been spiit into several tokens. In order to align the labels we need a function. Fortunately Huggingface has provided one which we can see below

In [119]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i) # This gives the same word id for words that have been split up
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx: # Only label the first token of a given word
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [118]:
len(doc_ids_i), len(interventions_tokens)

(4746, 4746)

In [115]:
data_dict = {
    "train": {
        "tokens"

AttributeError: 'list' object has no attribute 'get'